In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
gdf_kd = gpd.read_file("Data/rnpd.geojson")    #crs: epsg:4326

In [3]:
gdf_regije = gpd.read_file("Data/RazdelitevSlo/SR.geojson")

In [4]:
gdf_poplave_zelo_redke = gpd.read_file("Data/DRSV_OPKP_ZR_POPL/DRSV_OPKP_ZR_POPL.shp")
gdf_poplave_redke = gpd.read_file("Data/DRSV_OPKP_REDKE_POPL/DRSV_OPKP_REDKE_POPL.shp")
gdf_poplave_pogoste = gpd.read_file("Data/DRSV_OPKP_POGOSTE_POPL/DRSV_OPKP_POGOSTE_POPL.shp")
gdf_pozarna_ogrozenost = gpd.read_file("Data/pozarna_ogrozenost_majhen_100m.geojson")
gdf_plazovi = gpd.read_file("Data/DRSV_Opk_Plazovi_skupna/plazovi2.shp")

gdf_zrak_postaje = gpd.read_file("Data/zrak_postaje.geojson")

In [5]:
gdfs = [gdf_kd, gdf_regije, gdf_poplave_zelo_redke, gdf_poplave_redke, gdf_poplave_pogoste, gdf_pozarna_ogrozenost, gdf_plazovi, gdf_zrak_postaje]

In [6]:
gdf_regije = gdf_regije.to_crs(epsg=4326)
gdf_poplave_zelo_redke = gdf_poplave_zelo_redke.to_crs(epsg=4326)
gdf_poplave_redke = gdf_poplave_redke.to_crs(epsg=4326)
gdf_poplave_pogoste = gdf_poplave_pogoste.to_crs(epsg=4326)
gdf_pozarna_ogrozenost = gdf_pozarna_ogrozenost.to_crs(epsg=4326)
gdf_plazovi = gdf_plazovi.to_crs(epsg=4326)
gdf_zrak_postaje = gdf_zrak_postaje.to_crs(epsg=4326)

In [7]:
gdf_poplave_pogoste = gdf_poplave_pogoste.rename(columns={"PP_ID": "ID", "PP_IME": "IME"})
gdf_poplave_redke = gdf_poplave_redke.rename(columns={"RP_ID": "ID", "RP_IME": "IME"})
gdf_poplave_zelo_redke = gdf_poplave_zelo_redke.rename(columns={"ZR_ID": "ID", "ZR_IME": "IME"})

#popravimo da so isti tipi vseh stolpcev
gdf_poplave_zelo_redke["OC_ZAN"] = pd.to_numeric(gdf_poplave_zelo_redke["OC_ZAN"], errors="coerce")
gdf_poplave_zelo_redke["DAT_KON"] = pd.to_datetime(gdf_poplave_zelo_redke["DAT_KON"], errors="coerce")
gdf_poplave_zelo_redke["DATP_KON"] = pd.to_datetime(gdf_poplave_zelo_redke["DATP_KON"], errors="coerce")

gdf_poplave_zelo_redke["OBJECTID_1"] = pd.NA

gdf_poplave_pogoste["flood_level"] = 3       
gdf_poplave_redke["flood_level"] = 2         
gdf_poplave_zelo_redke["flood_level"] = 1

skupaj_poplave = gpd.GeoDataFrame(
    pd.concat(
        [gdf_poplave_pogoste, gdf_poplave_redke, gdf_poplave_zelo_redke],
        ignore_index=True
    ),
    crs=gdf_poplave_pogoste.crs
)

In [8]:
def spatial_join_attr(base, polygons, col):
    joined = base.sjoin(polygons[['geometry', col]], how='left', predicate='within')
    return joined.groupby(level=0)[col].first()    #groupy daa je le en rezultat ce pade v vec polygonov

In [9]:
gdf_kd['poplave_ocena'] = None
gdf_kd['poplave_ocena'] = gdf_kd['poplave_ocena'].astype(float)

gdf_kd['poplave'] = pd.NA
gdf_kd['poplave'] = spatial_join_attr(gdf_kd, skupaj_poplave, 'flood_level')
gdf_kd['pozar'] = spatial_join_attr(gdf_kd, gdf_pozarna_ogrozenost, 'pozar')
gdf_kd['plazovi'] = spatial_join_attr(gdf_kd, gdf_plazovi, 'DN')

In [10]:
gdf_kd = gdf_kd.to_crs(epsg=3857)
skupaj_poplave = skupaj_poplave.to_crs(epsg=3857)

nearest = gpd.sjoin_nearest(
    gdf_kd[gdf_kd['poplave'].isna()][['geometry']],  # only geometry, nothing else
    skupaj_poplave[['flood_level', 'geometry']],
    max_distance=1000,
    distance_col='razdalja_pop',
    how='left'
)

gdf_kd.loc[nearest.index, 'poplave'] = nearest['flood_level'].values

In [11]:
gdf_pozarna_ogrozenost = gdf_pozarna_ogrozenost.to_crs(epsg=3857)
gdf_kd['pozar'] = pd.to_numeric(gdf_kd['pozar'])

nearest = gpd.sjoin_nearest(
    gdf_kd[gdf_kd['pozar'].isna()][['geometry']],
    gdf_pozarna_ogrozenost[['pozar', 'geometry']],
    max_distance=1000,
    distance_col='razdalja_pop',
    how='left'
)

gdf_kd.loc[nearest.index, 'pozar'] = nearest['pozar'].values

TypeError: Invalid value '<StringArray>
['4', '4', '4', '3', '3', '3', '3', '4', '3', '3',
 ...
 '3', '3', '3', '4', nan, nan, '2', '4', '4', '4']
Length: 28450, dtype: str' for dtype 'float64'

In [ ]:
gdf_kd = gdf_kd.to_crs(epsg=4326)

In [ ]:
gdf_kd['regija'] = spatial_join_attr(gdf_kd, gdf_regije, 'SR_UIME')

In [ ]:
gdf_kd.loc[gdf_kd['regija'].isna(), 'regija'] = "Obalno-kraška"   #vse, ki so bile nan so v tržaškem zalivu/na obali

In [ ]:
gdf_kd['pozar'] = 5 - gdf_kd['pozar']

In [ ]:
gdf_kd['poplave'].isna().sum()